# 06 — Multi-site Wytham + Ofental

**Hypothesis.** Adding Ofental (temperate conifer) crops improves val-tile F1 / generalization.

**Runtime.** CPU for download/prepare; A100 for train

**Config.** `configs/experiment/multisite_wytham_ofental.yaml`

Train → frozen shared benchmark (`configs/experiment/compare_val.yaml`) → comparison table → optional Colab `runtime.unassign()`.

For tropical Robson/Litchfield, download those sites, combine into `data/multisite_tropical`, and set `TRAIN_CONFIG` to `configs/experiment/multisite_tropical.yaml`.


## 0. Options


In [ ]:
# Colab: mount Drive / clone repo first (see notebooks/00_colab.ipynb), then open this notebook from the repo.
TRAIN_CONFIG = "configs/experiment/multisite_wytham_ofental.yaml"
COMPARE_CONFIG = "configs/experiment/compare_val.yaml"
SKIP_TRAIN = False          # True to only re-benchmark an existing results/<run>/best.safetensors
AUTO_SHUTDOWN = True        # releases the Colab VM after results are written (set False while debugging)
CROP_ONLY_BENCHMARK = False  # True for Cedar Key (no Wytham-style tiles)


## 1. Repo root + VRAM probe


In [ ]:
import os, sys
from pathlib import Path
root = Path.cwd().resolve()
for _ in range(4):
    if (root / "src" / "treeseg").is_dir():
        break
    root = root.parent
os.chdir(root)
sys.path.insert(0, "src")

from treeseg.engine.experiment_run import ensure_repo_root, probe_resources, reset_cuda_peaks
import torch

print("cwd:", ensure_repo_root())
reset_cuda_peaks()
print("device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU")
probe_resources("notebook_start")


## 1b. Data prepare (CPU-friendly; skip if already on Drive)


In [ ]:
# Run on a CPU runtime first if data is missing, then reconnect with A100 for section 2.
from pathlib import Path
import subprocess, sys

def sh(cmd):
    print("+", cmd, flush=True)
    subprocess.check_call(cmd, shell=True)

if not Path("data/ofental_isprs/crops/train").is_dir():
    sh(f"{sys.executable} scripts/download_cherlet_site.py ofental")
    sh(f"{sys.executable} scripts/prepare_data.py configs/data/isprs_ofental.yaml")
if not Path("data/wytham_isprs/crops/train").is_dir():
    sh(f"{sys.executable} scripts/download_cherlet_site.py wytham")
    sh(f"{sys.executable} scripts/prepare_data.py configs/data/isprs_wytham_colab.yaml")
if not Path("data/multisite_wytham_ofental/crops/train").is_dir():
    sh(f"{sys.executable} scripts/combine_crop_sets.py --out data/multisite_wytham_ofental "
       "--train data/wytham_isprs/crops/train data/ofental_isprs/crops/train "
       "--val data/wytham_isprs/crops/val")
print("multi-site crops ready")


## 2. Train + shared benchmark + compare


In [ ]:
from treeseg.engine.experiment_run import run_train_and_compare
from IPython.display import display

table = run_train_and_compare(
    TRAIN_CONFIG,
    COMPARE_CONFIG,
    skip_train=SKIP_TRAIN,
    auto_shutdown=AUTO_SHUTDOWN,
    crop_only_benchmark=CROP_ONLY_BENCHMARK,
)
display(table)


## Compute notes

After the first successful run, inspect `results/<run>/resource_probe.json` for peak GPU GB. If peak is well under 14 GB, later LoRA twins may use a cheaper GPU than A100. Always keep checkpoints on Drive before `AUTO_SHUTDOWN`.
